# Mechanical Compression Analysis
**Perez et al. 2025 — Mycotecture Phase II**

**Dataset:** `input_data/` — OW1 and OW2 specimen groups

**Contents:**
1. [Setup](#setup)
2. [Single-Specimen Strength Analysis](#single-strength)
3. [Single-Specimen Modulus of Elasticity (MOE)](#single-moe)
4. [Per-Group Strength &amp; MOE Analysis](#multi-group)
5. [Bulk Overview — All Groups](#bulk)


## 1. Setup <a name='setup'></a>

In [ ]:
import os, sys
import pandas as pd
import numpy as np

# ── Set the repo root to the folder containing input_data/ and custom_python_functions/ ──
repo_root         = os.path.dirname(os.path.abspath("__file__"))   # update if running from elsewhere
python_files_root = os.path.join(repo_root, "custom_python_functions")

if python_files_root not in sys.path:
    sys.path.insert(0, python_files_root)

from Utility.csv_file_browser import reload_all_modules, force_fresh_import
reload_all_modules(clear_cache=True)

def try_reload(alias, module_name, fname):
    try:
        return force_fresh_import(module_name, os.path.join(python_files_root, fname))
    except Exception as e:
        print(f"❌ {alias}: {e}")
        return None

sp   = try_reload("sp",   "Plotting.Single_Plot",       "Plotting/Single_Plot.py")
mult = try_reload("mult", "Plotting.Multi_Plot",        "Plotting/Multi_Plot.py")
pp   = try_reload("pp",   "Plotting.Polyfit_Processor", "Plotting/Polyfit_Processor.py")
mp   = try_reload("mp",   "Plotting.Max_Processor",     "Plotting/Max_Processor.py")
cb   = try_reload("cb",   "Utility.csv_file_browser",   "Utility/csv_file_browser.py")
moe  = try_reload("moe",  "Plotting.MOE_Processor",     "Plotting/MOE_Processor.py")

target_folder = os.path.join(repo_root, "input_data")
print("✅ Ready. Data folder:", target_folder)


## 2. Single-Specimen Strength Analysis <a name='single-strength'></a>

Browse and select a CSV, inspect the raw curve, fit a polynomial,
extract yield stress and inflection markers, and export SVGs.


In [ ]:
# Browse and select a single specimen CSV
cb.browse_csv_files_interactive()


In [ ]:
# Load the selected specimen and plot raw stress-strain curve
df        = cb.browse_csv_files_interactive.selected_df
file_path = cb.browse_csv_files_interactive.selected_file_path

sp.plot_from_csv_results(df)


In [ ]:
# Fit polynomial to the stress-strain curve
df = pp.find_and_fit(df, file_path, overwrite=True)
display(df)


In [ ]:
# Compute yield stress and inflection points
mp.print_max_summary_df(df)


In [ ]:
# Interactive plot with yield stress and inflection markers
sp.plot_from_summary_row(df, markers={"inflection", "max"})


In [ ]:
# Configure export axis limits (Auto per-plot or Manual fixed)
import ipywidgets as widgets
export_limits = sp.make_export_axis_widget(df)


In [ ]:
# Export all specimens to individual strength SVGs
output_folder = os.path.join(repo_root, "compression_single_strength_plot", "output data")
os.makedirs(output_folder, exist_ok=True)

sp.export_all_strength_svg(
    target_folder,
    output_folder=output_folder,
    xlim=export_limits.xlim,
    ylim=export_limits.ylim,
)


## 3. Single-Specimen Modulus of Elasticity <a name='single-moe'></a>

Select a specimen and inspect its MOE fit, then export all MOE SVGs.


In [ ]:
# Browse and select a single specimen CSV (re-uses previous selection if already run)
cb.browse_csv_files_interactive()


In [ ]:
# Plot MOE fit for the selected specimen
df        = cb.browse_csv_files_interactive.selected_df
file_path = cb.browse_csv_files_interactive.selected_file_path

moe.plot_moe_from_df(
    df,
    best_window=True,
    fit_range=5,
    scan_min=0,
    scan_max=8,
    show_fit=True,
    xlim=(0, 25),
)


In [ ]:
# Export all specimens to individual MOE SVGs
output_folder = os.path.join(repo_root, "compression_single_MoE_plot", "output data")
os.makedirs(output_folder, exist_ok=True)

moe.export_all_moe_svg(
    target_folder,
    output_folder=output_folder,
    best_window=True,
    fit_range=5,
    scan_min=0,
    scan_max=8,
    xlim=(0, 25),
)


## 4. Per-Group Strength &amp; MOE Analysis <a name='multi-group'></a>

Batch-process all specimens, summarise by group, inspect interactively,
and export per-group overlay SVGs.


In [ ]:
# Build file index from the data folder
df_index = cb.build_df_index(target_folder)
display(df_index)


In [ ]:
# Interactive per-group stress-strain plot with strength markers
mult.select_subfolder_and_plot_group(df_index, markers={"max", "inflection"})


In [ ]:
# Batch-process strength metrics and save CSV summary
output_folder = os.path.join(repo_root, "compression_multi_strength_plot", "output data")
os.makedirs(output_folder, exist_ok=True)

df_index_strength = mp.batch_process_folder(target_folder, overwrite_cache=False)
display(df_index_strength)
df_index_strength.to_csv(os.path.join(output_folder, "strength_summary.csv"), index=False)
print("✅ Saved strength_summary.csv")


In [ ]:
# Summarise yield stress and inflection by group
mp.summarize_by_group_interactive(df_index_strength)


In [ ]:
# Batch-process MOE metrics and save CSV summary
df_index_moe = moe.batch_process_folder(
    target_folder, fit_range=5, scan_min=0, scan_max=8,
    best_window=True, overwrite_cache=False
)
display(df_index_moe)
df_index_moe.to_csv(os.path.join(output_folder, "moe_summary.csv"), index=False)
print("✅ Saved moe_summary.csv")


In [ ]:
# Interactive per-group MOE plot
mult.select_subfolder_and_plot_group_moe(df_index_moe)


In [ ]:
# Export per-group strength SVGs (all curves overlaid per group)
sp.export_groups_strength_svg(target_folder, output_folder=output_folder)

# Export per-group MOE SVGs
moe.export_groups_moe_svg(
    target_folder,
    output_folder=output_folder,
    best_window=True,
    fit_range=5,
    scan_min=0,
    scan_max=8,
    xlim=(0, 25),
)
print("✅ Group SVGs exported to:", output_folder)


## 5. Bulk Overview — All Groups <a name='bulk'></a>

Interactive dual-axis chart showing mean ± std strength and MOE
for every specimen group side-by-side.


In [ ]:
# Ensure both batch results are available (re-run if needed)
if 'df_index_strength' not in dir() or 'df_index_moe' not in dir():
    df_index_strength = mp.batch_process_folder(target_folder, overwrite_cache=False)
    df_index_moe      = moe.batch_process_folder(
        target_folder, fit_range=5, scan_min=0, scan_max=8,
        best_window=True, overwrite_cache=False
    )

output_folder = os.path.join(repo_root, "compression_bulk_data_plot", "output data")
os.makedirs(output_folder, exist_ok=True)

mult.interactive_strength_moe_by_group(
    df_index_strength, df_index_moe, output_folder=output_folder
)
